In [3]:
import os
import sys
import json
import shutil
import pandas as pd
from tqdm import tqdm
from textblob import TextBlob
from rdflib import Graph, Namespace, RDFS
from os.path import normpath, basename

# ================================
# CONFIGURAÇÕES DO AMBIENTE (Adjusted to use specific paths)
# ================================
# Define the base paths as provided by the user
FILES_LIST_DIR = '/home/ematos/phd/pipeline_city/DATASET_CITY_PROCESSED'
INPUT_FILES_DIR = '/home/ematos/phd/pipeline_city/DATASET_CITY'
OUTPUT_ROOT_DIR = '/home/ematos/phd/pipeline_city/INITIAL_NERS/OUTPUT_RESULTS'
DBPEDIA_OWL_PATH = '/home/ematos/phd/pipeline_city/models/dbpedia.owl'


# ================================
# RDF LOAD
# ================================
g = Graph()
try:
    g.parse(DBPEDIA_OWL_PATH)
except Exception as e:
    print(f"Error loading dbpedia.owl from {DBPEDIA_OWL_PATH}: {e}. Please ensure the file exists and the path is correct.")
    sys.exit(1)

DBO = Namespace("http://dbpedia.org/ontology/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

# ================================
# CACHE
# ================================
cache = {}
# Assuming cache.json is global and will be in the script's execution directory
if os.path.exists("cache.json"):
    with open("cache.json", "r") as f:
        cache = json.load(f)

# ================================
# TRADUÇÃO DE TIPOS DBPEDIA
# ================================
label_pt_dict = {
    'Person': 'Pessoa', 'Place': 'Lugar', 'Organisation': 'Organização',
    'Agent': 'Agente', 'Work': 'Obra', 'Species': 'Espécie', 'Artist': 'Artista',
    'Writer': 'Escritor', 'Company': 'Empresa', 'Cidade': 'Cidade', 'Country': 'País'
}

def traduz_tipo(tipo):
    return label_pt_dict.get(tipo, tipo)

# ================================
# FUNÇÕES DE CONSULTA LOCAL
# ================================
def get_local_type_and_superclass(label, lang="en"):
    matches = set()
    for s, p, o in g.triples((None, RDFS.label, None)):
        if str(o).strip().lower() == label.lower():
            for _, _, tipo in g.triples((s, RDF.type, None)):
                if str(tipo).startswith(str(DBO)):
                    matches.add(tipo)
    return list(matches)

def get_superclass_local(subclass_name):
    subclass_uri = DBO[subclass_name]
    current = subclass_uri
    while True:
        superclasses = list(g.objects(current, RDFS.subClassOf))
        if not superclasses:
            break
        superclass = superclasses[0]
        if not str(superclass).startswith(str(DBO)):
            break
        current = superclass
        if current == DBO.Agent:
            break
    return basename(normpath(str(current)))

def get_type(string):
    global cache
    string = string.strip().replace('"', '').title()
    if len(string) < 3:
        return "NOTFOUND"

    if string in cache:
        return cache[string]

    types = get_local_type_and_superclass(string)
    if types:
        tipo_uri = str(types[0])
        tipo = basename(normpath(tipo_uri))
        tipo_pt = traduz_tipo(tipo)
        superclass = get_superclass_local(tipo)
        superclass_pt = traduz_tipo(superclass)
        result = f"{tipo_pt};{superclass_pt}"
        cache[string] = result
        return result
    else:
        return "NOTFOUND"

# ================================
# TOKENIZAÇÃO E BIO-TAGGING
# ================================
def init_from_dataframe(df_input, tokens_list, df_translate):
    data = []
    count = 0
    for idx, row in df_input.iterrows():
        frase_pt = str(row['frase'])
        linha_id = row['linha']
        try:
            frase_en = str(TextBlob(frase_pt).translate(to='en'))
        except Exception:
            frase_en = frase_pt
        df_translate.loc[idx, 'frase_pt'] = frase_pt
        df_translate.loc[idx, 'frase_en'] = frase_en
        df_translate.loc[idx, 'linha'] = linha_id
        blob = TextBlob(frase_en)
        for sentence in blob.sentences:
            for word in sentence.words:
                data.append([count, word, "O", frase_pt, linha_id])
                tokens_list.append(word)
                count += 1
            data.append([count, ".", "O", frase_pt, linha_id])
            tokens_list.append(".")
            count += 1
    df_out = pd.DataFrame(data, columns=['n', 'Token', 'BIO', 'Frase_PT', 'Linha_ID'])
    df_out.set_index("n", inplace=True)
    return df_out

def seqnwords(df, tokens_list, n):
    total = len(tokens_list)
    for pos in tqdm(range(total - n + 1)):
        if pos % 5000 == 0:
            percent = (pos / total) * 100
            json.dump(cache, open("cache.json", 'w'))
        seq = " ".join(tokens_list[pos:pos + n]).strip()
        try:
            tag = get_type(seq)
        except Exception:
            tag = "NOTFOUND"
        if tag != "NOTFOUND":
            if n == 1:
                df.at[pos, 'BIO'] = "U-" + tag
            else:
                df.at[pos, 'BIO'] = "B-" + tag
                for i in range(1, n - 1):
                    df.at[pos + i, 'BIO'] = "I-" + tag
                df.at[pos + n - 1, 'BIO'] = "L-" + tag

def rebuild_xml(tokens, tags):
    resultado = []
    inside_tag = False
    tag_atual = ""
    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            tipo = tag[2:].split(";")[0].upper().replace(" ", "_")
            resultado.append(f"<{tipo}>{token}")
            inside_tag = True
            tag_atual = tipo
        elif tag.startswith("I-") and inside_tag:
            resultado.append(f" {token}")
        elif tag.startswith("L-") and inside_tag:
            resultado.append(f" {token}</{tag_atual}>")
            inside_tag = False
        elif tag.startswith("U-"):
            tipo = tag[2:].split(";")[0].upper().replace(" ", "_")
            resultado.append(f"<{tipo}>{token}</{tipo}>")
        else:
            if inside_tag:
                resultado.append(f"</{tag_atual}>")
                inside_tag = False
            resultado.append(token)
    if inside_tag:
        resultado.append(f"</{tag_atual}>")
    return " ".join(resultado)

def rebuild_annotated_sentences(df_tokens):
    frases_annotadas = []
    frase_atual = []
    bio_atual = []
    frase_original = None
    linha_atual = None

    for _, row in df_tokens.iterrows():
        token = row['Token']
        bio = row['BIO']
        frase = row['Frase_PT']
        linha = row['Linha_ID']

        if frase_original is None:
            frase_original = frase
            linha_atual = linha

        if frase != frase_original:
            frase_annotada = rebuild_xml(frase_atual, bio_atual)
            frases_annotadas.append((linha_atual, frase_original, frase_annotada))
            frase_atual = []
            bio_atual = []
            frase_original = frase
            linha_atual = linha

        frase_atual.append(token)
        bio_atual.append(bio)

    if frase_atual:
        frase_annotada = rebuild_xml(frase_atual, bio_atual)
        frases_annotadas.append((linha_atual, frase_original, frase_annotada))

    return pd.DataFrame(frases_annotadas, columns=['linha', 'frase_pt', 'frase_annotada_xml'])

def process_dataframe(input_df, output_file, max_n=7):
    global cache
    tokens_list = []
    df_translate = pd.DataFrame()
    df = init_from_dataframe(input_df, tokens_list, df_translate)
    for n in tqdm(range(1, max_n + 1)):
        print(f"Processando {n}-gramas...")
        seqnwords(df, tokens_list, n)
        df.to_csv(f"{output_file}_{n}.csv", index=True)
        json.dump(cache, open("cache.json", 'w'))

    df.to_csv(f"{output_file}.csv", index=True)
    df_translate.to_csv(f"{output_file}_translated.csv", index=False)

    df_annotated = rebuild_annotated_sentences(df)
    df_annotated.to_csv(f"{output_file}_xml.csv", index=False)
    json.dump(cache, open("cache.json", 'w'))

# ================================
# UTILITÁRIOS
# ================================
def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

# ================================
# EXECUÇÃO PRINCIPAL (Modified to handle input file format)
# ================================
def main():
    try:
        # Read the list of files to process from the specified path
        files_list_path = os.path.join(FILES_LIST_DIR, 'files_list.csv')
        lista = pd.read_csv(files_list_path)
    except Exception as e:
        print(f"Erro lendo {files_list_path}: {e}")
        sys.exit(1)

    for _, row in lista.iterrows():
        filename = row['filename']
        basename_file = row['basename']

        input_path = os.path.join(INPUT_FILES_DIR, filename)
        output_dir = os.path.join(OUTPUT_ROOT_DIR, basename_file)
        output_file = os.path.join(output_dir, "output_dbpediaNER")

        print(f"\n--- Processando {filename} ---")
        ensure_directory_exists(output_dir)

        try:
            # Modified to read without header and assign 'frase'
            df_input = pd.read_csv(input_path, sep='\t', header=None, names=['frase'])
            # Add 'linha' column, using the DataFrame index as line numbers
            df_input['linha'] = df_input.index
        except FileNotFoundError:
            print(f"Erro: Arquivo de entrada '{input_path}' não encontrado. Pulando para o próximo.")
            continue
        except Exception as e:
            print(f"Erro lendo arquivo {input_path}: {e}")
            continue

        # The check for 'linha' and 'frase' columns is no longer strictly necessary
        # because we are now explicitly creating them, but keeping it for robustness
        # in case of unexpected issues during loading.
        if 'linha' not in df_input.columns or 'frase' not in df_input.columns:
            print(f"Arquivo {filename} não possui colunas 'linha' e 'frase' após o pré-processamento. Pulando para o próximo.")
            continue

        process_dataframe(df_input, output_file, max_n=7)
        print(f"Saída salva em {output_dir}")

        # Move generated output files to the specific output_dir.
        # This assumes the script creates files starting with 'output_dbpediaNER' in the current directory
        # before moving them.
        for file in [f for f in os.listdir('.') if os.path.isfile(f) and f.startswith('output_dbpediaNER')]:
            shutil.move(file, os.path.join(output_dir, file))
        
        # Save cache after each file processing, assuming cache is global.
        with open("cache.json", "w") as cache_file:
            json.dump(cache, cache_file)

if __name__ == "__main__":
    main()


--- Processando Peru.txt ---


  0%|                                                     | 0/7 [00:00<?, ?it/s]

Processando 1-gramas...



 14%|██████▍                                      | 1/7 [00:15<01:33, 15.53s/it]

Processando 2-gramas...


...

Processando 6-gramas...



 86%|██████████████████████████████████████▌      | 6/7 [01:48<00:19, 19.48s/it]

Processando 7-gramas...



100%|█████████████████████████████████████████████| 7/7 [02:09<00:00, 18.48s/it]


Saída salva em /home/ematos/phd/pipeline_city/INITIAL_NERS/OUTPUT_RESULTS/Port_Louis
